# Fase 3 — Notebook 01: Extração e Filtragem ELSI-Brasil

**Objetivo:** ler os 8 arquivos do Censo Demográfico 2022, filtrar apenas os setores
censitários pertencentes aos **70 municípios da amostra do ELSI-Brasil** e exportar
uma base bruta unificada, pronta para a análise exploratória do Notebook 02.

**Por que existe:** os notebooks anteriores (Fase 1 e Fase 2) processavam o Brasil
inteiro (~468 mil setores de 5.297 municípios). O objetivo científico do projeto é
uma análise **intraurbana** restrita aos 70 municípios do ELSI-Brasil — este notebook
aplica esse filtro pela primeira vez na pipeline.

**Entradas:**
- `dados/Agregados_por_setores_*.csv` — 8 CSVs do Censo 2022 (IBGE)
- `dados/municipios_elsi_brasil.csv` — lista oficial dos 70 municípios ELSI

**Saída:**
- `banco_de_dados/Base_ELSI_Bruta_Censo2022.csv` — base bruta filtrada, com sigilo
  preservado (marcação `X` do IBGE mantida).

**Etapas:**
1. Carregar a lista dos 70 municípios ELSI.
2. Ler o arquivo básico do Censo e identificar os setores ELSI (cruzamento por UF +
   nome normalizado).
3. Validar que todos os 70 municípios foram encontrados.
4. Ler os outros 7 arquivos do Censo filtrando por `CD_SETOR` ∈ lista ELSI.
5. Fazer o merge unificado.
6. Classificar morfologia urbana predominante por setor.
7. Auditoria de integridade.
8. Exportar a base bruta filtrada.

## 1. Imports, caminhos e funções utilitárias

In [ ]:
import os
import unicodedata
import pandas as pd
import numpy as np

# Caminhos relativos ao notebook (notebooks/Fase3_EDA_ELSI/)
CAMINHO_DADOS = '../../dados/'
CAMINHO_BD = '../../banco_de_dados/'
os.makedirs(CAMINHO_BD, exist_ok=True)


def normalize_name(s):
    """Lowercase, sem acentos, espaços colapsados. Para cruzar nomes de municípios."""
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return ' '.join(s.split())


def ler_csv_padronizado(caminho, usecols, rename_cols=None,
                       encoding_list=('utf-8', 'latin1', 'cp1252'),
                       sep=';', dtype=str):
    """Lê um CSV testando múltiplos encodings; renomeia colunas-chave se necessário."""
    for enc in encoding_list:
        try:
            df = pd.read_csv(caminho, sep=sep, dtype=dtype, usecols=usecols,
                             encoding=enc, low_memory=False)
            if rename_cols:
                df = df.rename(columns=rename_cols)
            print(f'  {os.path.basename(caminho)} — encoding: {enc} — {len(df):,} linhas')
            return df
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f'Não foi possível ler {caminho}.')

print('Setup concluído.')

## 2. Carregar a lista dos 70 municípios ELSI-Brasil

Fonte: <https://elsi.cpqrr.fiocruz.br/amostra/>. A lista foi consolidada em
`dados/municipios_elsi_brasil.csv` com as colunas: `regiao`, `uf_codigo` (código IBGE
de 2 dígitos), `uf_sigla`, `nm_municipio`.

In [ ]:
df_elsi = pd.read_csv(CAMINHO_DADOS + 'municipios_elsi_brasil.csv', sep=';', dtype=str)
df_elsi['nm_municipio_norm'] = df_elsi['nm_municipio'].map(normalize_name)
df_elsi['chave_municipio'] = df_elsi['uf_codigo'].str.zfill(2) + '|' + df_elsi['nm_municipio_norm']

assert len(df_elsi) == 70, f'Esperado 70 municípios ELSI, encontrado {len(df_elsi)}'
assert df_elsi['chave_municipio'].is_unique, 'Chave (UF + nome) tem duplicatas — verificar lista.'

print(f'{len(df_elsi)} municípios ELSI-Brasil carregados.')
print('\nDistribuição por região:')
print(df_elsi['regiao'].value_counts().to_string())
print('\nDistribuição por UF:')
print(df_elsi.groupby(['regiao', 'uf_sigla']).size().to_string())

## 3. Identificar os setores ELSI no arquivo básico

O arquivo básico (`Agregados_por_setores_basico_BR_*.csv`) contém um registro por setor
censitário do país (~468 mil) e tem as colunas `CD_SETOR` (chave) e `NM_MUN`. A partir
do `CD_SETOR` derivamos o **código IBGE da UF** (primeiros 2 dígitos) e do **município**
(primeiros 7 dígitos). O cruzamento com a lista ELSI é feito pela chave composta
`(uf_codigo, nm_municipio_normalizado)` — necessário porque há municípios homônimos em
UFs diferentes (ex.: Tabatinga em AM e SP).

In [ ]:
col_basico = ['CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO', 'v0001']

print('Lendo o arquivo básico do Censo 2022 (~468 mil setores)...')
df_basico = ler_csv_padronizado(
    CAMINHO_DADOS + 'Agregados_por_setores_basico_BR_20250417.csv',
    usecols=col_basico,
)

# Derivar UF e código de município a partir do CD_SETOR (15 dígitos: UF[2] + MUN[5] + ...)
df_basico['CD_UF'] = df_basico['CD_SETOR'].str[:2]
df_basico['CD_MUN'] = df_basico['CD_SETOR'].str[:7]
df_basico['NM_MUN_NORM'] = df_basico['NM_MUN'].map(normalize_name)
df_basico['chave_municipio'] = df_basico['CD_UF'] + '|' + df_basico['NM_MUN_NORM']

print(f'\nTotal de setores no Censo 2022: {len(df_basico):,}')
print(f'Total de municípios no Censo 2022: {df_basico["CD_MUN"].nunique():,}')

In [ ]:
# Filtrar setores cujo município está na lista ELSI
chaves_elsi = set(df_elsi['chave_municipio'])
df_basico_elsi = df_basico[df_basico['chave_municipio'].isin(chaves_elsi)].copy()

setores_elsi = set(df_basico_elsi['CD_SETOR'])
muns_encontrados = df_basico_elsi['chave_municipio'].unique()

print(f'Setores filtrados: {len(df_basico_elsi):,}')
print(f'Municípios ELSI encontrados: {len(muns_encontrados)} de 70')
print(f'Municípios distintos na base filtrada: {df_basico_elsi["CD_MUN"].nunique()}')

### Validação — todos os 70 municípios ELSI foram localizados?

Se algum município não foi encontrado, a célula abaixo lista os faltantes (geralmente
indicam grafia divergente entre a lista ELSI e o IBGE — corrigir em
`dados/municipios_elsi_brasil.csv`).

In [ ]:
faltantes = df_elsi[~df_elsi['chave_municipio'].isin(set(muns_encontrados))]

if len(faltantes) == 0:
    print('✅ Todos os 70 municípios ELSI foram localizados no Censo 2022.')
else:
    print(f'⚠️  {len(faltantes)} municípios ELSI NÃO foram encontrados:')
    print(faltantes[['regiao', 'uf_sigla', 'nm_municipio']].to_string(index=False))
    print('\nProváveis causas: grafia divergente, acento, hífen, ou nome alternativo.')
    print('Verifique a coluna NM_MUN do arquivo básico do Censo para a UF em questão.')

# Resumo de setores por município
resumo_mun = (df_basico_elsi.groupby(['CD_UF', 'NM_MUN'])
              .size().rename('n_setores').reset_index()
              .sort_values(['CD_UF', 'NM_MUN']))
print(f'\nResumo (primeiros 10 municípios por contagem de setores):')
print(resumo_mun.nlargest(10, 'n_setores').to_string(index=False))

## 4. Ler os demais 7 arquivos do Censo, filtrando por `CD_SETOR`

Como já temos o conjunto de `CD_SETOR` da ELSI, lemos cada um dos outros 7 arquivos e
filtramos imediatamente para reduzir o uso de memória. Cada arquivo é grande
(domicílio2 tem 747 MB, alfabetização 701 MB), por isso a filtragem é feita em
**chunks** de 100 mil linhas.

In [ ]:
# Mapeamento de colunas alvo por arquivo (mesmas variáveis usadas na Fase 2)
ARQUIVOS = {
    'dom1': {
        'arquivo': 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv',
        'usecols': ['CD_setor', 'V00001', 'V00002', 'V00005', 'V00006',
                    'V00047', 'V00048', 'V00049', 'V00050', 'V00051', 'V00052'],
        'rename': {'CD_setor': 'CD_SETOR'},
    },
    'dom2': {
        'arquivo': 'Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv',
        'usecols': ['setor', 'V00112', 'V00113', 'V00114', 'V00115', 'V00116', 'V00117', 'V00118',
                    'V00312', 'V00313', 'V00314', 'V00315', 'V00316',
                    'V00398', 'V00399', 'V00400', 'V00401', 'V00402',
                    'V00236', 'V00238'],
        'rename': {'setor': 'CD_SETOR'},
    },
    'alfab': {
        'arquivo': 'Agregados_por_setores_alfabetizacao_BR.csv',
        'usecols': ['CD_setor', 'V00900', 'V00901'],
        'rename': {'CD_setor': 'CD_SETOR'},
    },
    'raca': {
        'arquivo': 'Agregados_por_setores_cor_ou_raca_BR.csv',
        'usecols': ['CD_SETOR', 'V01318', 'V01320', 'V01321'],
        'rename': None,
    },
    'renda': {
        'arquivo': 'Agregados_por_setores_renda_responsavel_BR.csv',
        'usecols': ['CD_SETOR', 'V06004'],
        'rename': None,
    },
    'demog': {
        'arquivo': 'Agregados_por_setores_demografia_BR.csv',
        'usecols': ['CD_setor', 'V01031', 'V01032', 'V01033'],
        'rename': {'CD_setor': 'CD_SETOR'},
    },
    'parent': {
        'arquivo': 'Agregados_por_setores_parentesco_BR.csv',
        'usecols': ['CD_SETOR', 'V01042'],
        'rename': None,
    },
}


def ler_filtrado(arquivo, usecols, rename, setores_alvo,
                 encoding_list=('utf-8', 'latin1', 'cp1252'), chunksize=100_000):
    """Lê um CSV em chunks, filtrando linhas cujo CD_SETOR está em setores_alvo."""
    caminho = CAMINHO_DADOS + arquivo
    chave_origem = (rename and next(iter(rename))) or 'CD_SETOR'
    for enc in encoding_list:
        try:
            pedacos = []
            reader = pd.read_csv(caminho, sep=';', dtype=str, usecols=usecols,
                                 encoding=enc, chunksize=chunksize, low_memory=False)
            for chunk in reader:
                if rename:
                    chunk = chunk.rename(columns=rename)
                chunk = chunk[chunk['CD_SETOR'].isin(setores_alvo)]
                if len(chunk):
                    pedacos.append(chunk)
            df = pd.concat(pedacos, ignore_index=True) if pedacos else pd.DataFrame(columns=[chave_origem] + list(usecols))
            print(f'  {arquivo} — encoding: {enc} — {len(df):,} setores filtrados')
            return df
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f'Não foi possível ler {caminho}.')

print('Funções definidas.')

In [ ]:
print('Lendo e filtrando os 7 arquivos complementares (chunks de 100 mil linhas)...\n')

dfs = {}
for chave, cfg in ARQUIVOS.items():
    dfs[chave] = ler_filtrado(cfg['arquivo'], cfg['usecols'], cfg['rename'], setores_elsi)

print('\nLeitura concluída.')

## 5. Merge unificado pelo `CD_SETOR`

Junção `LEFT JOIN` mantendo todos os setores ELSI do arquivo básico. Se algum setor
estiver ausente nos arquivos complementares, fica com `NaN` nas colunas correspondentes
(será inspecionado na auditoria).

In [ ]:
df = df_basico_elsi.drop(columns=['NM_MUN_NORM', 'chave_municipio']).copy()
for chave in ['dom1', 'dom2', 'alfab', 'raca', 'renda', 'demog', 'parent']:
    df = df.merge(dfs[chave], on='CD_SETOR', how='left')
    print(f'  + {chave}: shape após merge = {df.shape}')

print(f'\nBase unificada: {len(df):,} setores × {len(df.columns)} colunas')

## 6. Classificador de Morfologia Urbana

Define o **tipo de moradia predominante** por setor a partir das contagens dos arquivos
do IBGE: casa (V00047), casa de vila/condomínio (V00048), apartamento (V00049),
cortiço (V00050), maloca indígena (V00051), estrutura degradada (V00052). Útil para
análises da EDA (perfis morfológicos vs. vulnerabilidade).

In [ ]:
DIC_MORFOLOGIA = {
    'V00047': 'Casa',
    'V00048': 'Casa de Vila/Condomínio',
    'V00049': 'Apartamento',
    'V00050': 'Cortiço/Casa de Cômodos',
    'V00051': 'Maloca Indígena',
    'V00052': 'Estrutura Degradada/Inacabada',
}

cols_morf = list(DIC_MORFOLOGIA)
morf_num = df[cols_morf].apply(pd.to_numeric, errors='coerce').fillna(0)
df['Moradia_Predominante'] = morf_num.idxmax(axis=1).map(DIC_MORFOLOGIA)
df.loc[morf_num.sum(axis=1) == 0, 'Moradia_Predominante'] = 'Indefinido/Sem Moradia'

print(df['Moradia_Predominante'].value_counts(dropna=False).to_string())

## 7. Auditoria de integridade

Verifica: (a) nenhum setor foi duplicado ou perdido no merge; (b) a chave primária está
íntegra; (c) o marcador de sigilo `X` do IBGE foi preservado (necessário para o
tratamento de elegibilidade nas etapas seguintes); (d) variáveis-chave das 8 fontes
estão presentes.

In [ ]:
erros = 0
print('=== AUDITORIA DE INTEGRIDADE ===\n')

if len(df) == len(df_basico_elsi):
    print(f'[OK] Linhas mantidas: {len(df):,}')
else:
    print(f'[ERRO] Linhas mudaram: esperado {len(df_basico_elsi):,}, obtido {len(df):,}')
    erros += 1

if df['CD_SETOR'].is_unique and df['CD_SETOR'].notna().all():
    print('[OK] CD_SETOR íntegra (única, sem nulos).')
else:
    print('[ERRO] CD_SETOR tem duplicatas ou valores nulos.')
    erros += 1

cols_essenciais = ['v0001', 'V00001', 'V00112', 'V00312', 'V00398',
                   'V00900', 'V00901', 'V01318', 'V06004', 'V01031',
                   'V01042', 'Moradia_Predominante']
faltando = [c for c in cols_essenciais if c not in df.columns]
if not faltando:
    print(f'[OK] Variáveis essenciais presentes ({len(cols_essenciais)} colunas).')
else:
    print(f'[ERRO] Colunas faltando: {faltando}')
    erros += 1

setores_com_sigilo = df.eq('X').any(axis=1).sum()
if setores_com_sigilo > 0:
    print(f'[OK] Sigilo do IBGE preservado: {setores_com_sigilo:,} setores contêm "X".')
else:
    print('[ALERTA] Nenhum sigilo "X" detectado — verificar se a leitura está correta.')

n_muns = df['CD_MUN'].nunique()
if n_muns == 70:
    print(f'[OK] 70 municípios ELSI presentes.')
else:
    print(f'[ALERTA] {n_muns} municípios distintos na base (esperado 70).')

print('\n' + ('🟢 BASE APROVADA.' if erros == 0 else f'🔴 {erros} ERROS — INTERROMPER.'))

## 8. Exportar a base bruta filtrada

Saída final: `banco_de_dados/Base_ELSI_Bruta_Censo2022.csv`. Mantém o sigilo `X`
intacto — o tratamento de sigilo e o cálculo dos indicadores ficam para etapas
posteriores (após a EDA do Notebook 02).

In [ ]:
saida = CAMINHO_BD + 'Base_ELSI_Bruta_Censo2022.csv'
df.to_csv(saida, index=False, sep=';', encoding='utf-8-sig')

tamanho_mb = os.path.getsize(saida) / (1024 * 1024)
print(f'✅ Base exportada para: {saida}')
print(f'   {len(df):,} setores × {len(df.columns)} colunas — {tamanho_mb:.1f} MB')

print('\nPróximo passo: abrir o Notebook 02 (Análises Descritivas).')